# 第十三课:SGD / Adam——参数更新的艺术

讲义:[13-optimizer.md](13-optimizer.md)。

In [ ]:
import numpy as np

class AdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):
        self.p, self.lr, self.eps, self.wd = params, lr, eps, wd
        self.b1, self.b2 = betas
        self.m = np.zeros_like(params)
        self.v = np.zeros_like(params)
        self.t = 0

    def step(self, grads):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * grads
        self.v = self.b2 * self.v + (1 - self.b2) * grads ** 2
        m_hat = self.m / (1 - self.b1 ** self.t)     # 偏差修正
        v_hat = self.v / (1 - self.b2 ** self.t)
        self.p = self.p - self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.wd * self.p)

# 玩具:w 从 5.0 收敛到 3.0
w = np.array([5.0])
opt = AdamW(w.copy(), lr=0.05, wd=0.0)
loss0 = (w[0] - 3) ** 2
for _ in range(300):
    g = 2 * (opt.p[0] - 3)
    opt.step(np.array([g]))
print('loss:', round(loss0, 1), '→', f'{(opt.p[0]-3)**2:.8f}', ' w =', round(opt.p[0], 6))

## 对照实验:去掉偏差修正(作业 C3)

In [ ]:
def run(with_bias_correction, steps=60):
    w = np.array([5.0]); m = v = np.array([0.0]); hist = []
    for t in range(1, steps + 1):
        g = 2 * (w[0] - 3)
        m = 0.9 * m + 0.1 * g
        v = 0.999 * v + 0.001 * g * g
        if with_bias_correction:
            mh, vh = m / (1 - 0.9 ** t), v / (1 - 0.999 ** t)
        else:
            mh, vh = m, v
        w = w - 0.05 * mh / (np.sqrt(vh) + 1e-8)
        hist.append((w[0] - 3) ** 2)
    return hist

a, b = run(True), run(False)
for t in (1, 5, 20, 60):
    print(f'第{t:3d}步  loss(有修正)={a[t-1]:.6f}   loss(无修正)={b[t-1]:.6f}')

## 生产账本(讲义 §4)

In [ ]:
for name, n in [('7B', 7e9), ('13B', 13e9), ('70B', 70e9)]:
    total = n * 12          # 权重2 + 梯度2 + m4 + v4 字节
    print(f'{name}: {total/1024**3:.1f} GB(其中优化器状态 {n*8/1024**3:.1f} GB)')
print('80GB 卡 ÷ 84GB(7B 训练态)→ 单卡放不下,这就是 ZeRO 的起点')

In [ ]:
# TODO(C4): 把第 12 课两层 MLP 接上 AdamW 训 500 步
# rng = np.random.default_rng(0)
# W1, W2 = ..., ...
# x, y = rng.normal(size=5), rng.normal(size=5)
# 循环: 前向 → 手写反向(dW1, dW2) → opt.step
# 记录 loss 曲线,期望: 0.2 左右降到 <0.01(随机性下量级即可)